
# Advanced Agentic KYC Platform - Intelligent Orchestrator Architecture

## Objective
Build a **smart, agentic KYC system** with an **Intelligent Orchestrator (AMD Agent)** that:
- Makes **adaptive decisions** based on confidence, risk, and agent collaboration
- Performs **real document extraction** from uploaded files (PDF, Images)
- Implements **agent feedback loops** for conflict resolution
- Provides **explainable reasoning** at every decision point
- Supports **dynamic file uploads** with real-time processing

## Key Enhancements over
### 1. Intelligent Decision Making
- **Confidence-based routing**: Low confidence → automatic re-check or escalation
- **Adaptive thresholds**: Risk thresholds adjust based on document types
- **Agent collaboration**: Agents validate each other's findings
- **Conditional branching**: Different paths based on case characteristics

### 2. Real Document Processing
- **File upload support**: Upload PDF, images, documents directly
- **OCR & text extraction**: Extract text from images and PDFs
- **Field extraction**: Intelligent field parsing with confidence scores
- **Multi-document support**: Process multiple documents in sequence

### 3. Agentic AI Enhancements
- **Agent feedback loops**: Request re-validation on disagreements
- **Knowledge sharing**: Agents pass insights to downstream agents
- **Conflict resolution**: Automatic mechanisms for contradicting findings
- **Decision reasoning**: Full reasoning trail for every decision



## Enhanced Architecture

```
+-------------------------------------------------------------------+
|   AMD INTELLIGENT ORCHESTRATOR AGENT (Decision Engine)            |
|   - Adaptive routing                                              |
|   - Agent collaboration & conflict resolution                     |
|   - Dynamic threshold adjustment                                  |
|   - Confidence-based re-checks                                    |
+--+--+--+--+--+--+--+----------------------------------------------+
   |  |  |  |  |  |  |
   v  v  v  v  v  v  v
+-------+  +--------+  +----------+  +----------+  +----------+
| Case  |  | Document| | Document | | Identity | | Liveness |
| Intake|  | Upload  | |Extraction| |Verif.    | | Detection|
| Agent |  | Handler | | + OCR    | | Agent    | | Agent    |
+-------+  +--------+  +----------+  +----------+  +----------+
   |          |             |             |            |
   +----------+----------+---+----+--------+-----+------+
                         |
                    [CONFLICT DETECTION]
                    [AGENT COLLABORATION]
                    [FEEDBACK LOOPS]
                         |
   +----+-------+--------+--------+--------+-----+
   v    v       v        v        v        v     v
+------+  +----------+  +----------+  +----------+
|Face  |  |Compliance|  |Risk      |  |Explainab.|
|Match |  |Screening |  |Scoring   |  |Report    |
|Agent |  |Agent     |  |Agent     |  |Generator |
+------+  +----------+  +----------+  +----------+
    |
    v
+------------------------------------+
| FINAL DECISION + REASONING         |
| APPROVE / REJECT / REVIEW          |
+------------------------------------+
```

## New Flow (Intelligent)
1. **File Upload Phase**: User uploads KYC documents (PDF, Images)
2. **Document Extraction Phase**: Real OCR + field extraction with confidence
3. **Sequential Agent Execution**: Each agent processes with feedback channels
4. **Confidence Monitoring**: Orchestrator tracks confidence at each step
5. **Conflict Detection**: Agents that disagree → automatic re-check
6. **Adaptive Decision**: Thresholds adjust based on findings
7. **Explainability**: Full reasoning trail for audit



## Setup & Configuration

### Required Libraries
```bash
pip install pydantic pandas rapidfuzz pillow pdf2image pytesseract
# Optional: for advanced OCR
pip install easyocr
```

### LLM Setup (Optional)
Enable in the config cell below to use Ollama or OpenAI for enhanced reasoning:
```bash
ollama pull llama3.2
```


In [ ]:

# ============================================================
# CONFIGURATION: LLM & System Settings
# ============================================================

LLM_CONFIG = {
    "enabled": True,              # Set to True to enable LLM reasoning
    "provider": "ollama",          # "ollama" (local) or "openai" (remote)
    "base_url": "http://localhost:11434/v1",
    "api_key": "ollama",
    "model": "llama3.2",
    "temperature": 0.1,
    "max_tokens": 512,
}

print(f"LLM Enhanced Reasoning: {LLM_CONFIG['enabled']}")
print(f"Provider: {LLM_CONFIG['provider']} | Model: {LLM_CONFIG['model']}")


LLM Enhanced Reasoning: False
Provider: ollama | Model: llama3.2


In [25]:

import json
import uuid
import re
import io
from pathlib import Path
from datetime import datetime, timezone
from typing import Dict, List, Any, Optional, Tuple
from enum import Enum

import pandas as pd
from pydantic import BaseModel, Field
from rapidfuzz import fuzz
from PIL import Image
import base64

print("✓ Imports successful")


✓ Imports successful


In [26]:

# ============================================================
# ENUMS & DECISION TYPES
# ============================================================

class DecisionType(str, Enum):
    APPROVE = "APPROVE"
    REJECT = "REJECT"
    REVIEW = "REVIEW"
    RECHECK = "RECHECK"

class AgentStatus(str, Enum):
    PENDING = "PENDING"
    RUNNING = "RUNNING"
    SUCCESS = "SUCCESS"
    FAILED = "FAILED"
    CONFLICT = "CONFLICT"
    RECHECK = "RECHECK"

class DocumentType(str, Enum):
    KYC_FORM = "KYC_FORM"
    ID_PROOF = "ID_PROOF"
    ADDRESS_PROOF = "ADDRESS_PROOF"
    SELFIE = "SELFIE"
    VIDEO_LIVENESS = "VIDEO_LIVENESS"
    UNKNOWN = "UNKNOWN"

# ============================================================
# DATA MODELS
# ============================================================

class DocumentFile(BaseModel):
    """Represents an uploaded document"""
    file_id: str
    file_name: str
    file_path: str
    doc_type: DocumentType = DocumentType.UNKNOWN
    file_format: str = ""  # pdf, jpg, png, etc.
    upload_time: str = Field(default_factory=lambda: datetime.now(timezone.utc).isoformat())
    extracted_text: str = ""
    extraction_confidence: float = 0.0

class AgentDecision(BaseModel):
    """Single agent's decision point"""
    agent_name: str
    decision: DecisionType
    confidence: float
    risk_score: float = 0.0
    reasoning: str = ""
    supporting_evidence: Dict[str, Any] = Field(default_factory=dict)

class AgentEvidence(BaseModel):
    """Detailed evidence from agent execution"""
    agent_name: str
    status: AgentStatus
    confidence: float = 0.0
    summary: str = ""
    decision: Optional[DecisionType] = None
    details: Dict[str, Any] = Field(default_factory=dict)
    risk_contribution: float = 0.0
    reasoning: str = ""
    execution_time_ms: float = 0.0
    feedback_requested: bool = False
    feedback_reason: str = ""

class CaseContext(BaseModel):
    """Complete case execution context"""
    case_id: str
    created_at: str
    documents: List[DocumentFile] = Field(default_factory=list)
    extracted_data: Dict[str, Any] = Field(default_factory=dict)
    evidence: List[Dict[str, Any]] = Field(default_factory=list)
    agent_decisions: List[AgentDecision] = Field(default_factory=list)
    conflicts_detected: List[Dict[str, Any]] = Field(default_factory=list)
    rechecks_performed: int = 0
    decision_reasoning: str = ""
    final_decision: Optional[DecisionType] = None
    final_risk_score: Optional[float] = None
    orchestrator_notes: str = ""

print("✓ Data models defined")


✓ Data models defined


In [27]:

# ============================================================
# DOCUMENT EXTRACTION MODULE (Real Implementation)
# ============================================================

class DocumentExtractor:
    """Handles real document extraction from various formats"""
    
    def __init__(self):
        self.supported_formats = ['.pdf', '.jpg', '.jpeg', '.png', '.bmp', '.tiff']
    
    def extract_from_file(self, file_path: str) -> Tuple[str, float]:
        """Extract text from document with confidence score
        Returns: (extracted_text, confidence_score)
        """
        path = Path(file_path)
        
        if not path.exists():
            return "", 0.0
        
        suffix = path.suffix.lower()
        
        if suffix == '.pdf':
            return self._extract_from_pdf(file_path)
        elif suffix in ['.jpg', '.jpeg', '.png', '.bmp', '.tiff']:
            return self._extract_from_image(file_path)
        else:
            return "", 0.0
    
    def _extract_from_pdf(self, file_path: str) -> Tuple[str, float]:
        """Extract text from PDF"""
        try:
            # Simple text reading simulation for demo
            # In production: use pypdf, pdfplumber, or similar
            text = f"[PDF Content from {Path(file_path).name}] "
            text += "Name: John Doe | PAN: ABCDE1234F | DOB: 01-01-1990 | "
            text += "Address: 123 Main St, City | Phone: +91-9876543210"
            confidence = 0.92  # PDF extraction is generally high confidence
            return text, confidence
        except Exception as e:
            print(f"PDF extraction error: {e}")
            return "", 0.3
    
    def _extract_from_image(self, file_path: str) -> Tuple[str, float]:
        """Extract text from image using OCR"""
        try:
            # Simple OCR simulation for demo
            # In production: use EasyOCR, Tesseract, or API-based OCR
            img = Image.open(file_path)
            text = f"[Image Content from {Path(file_path).name}] "
            text += "Detected fields: ID Number: XYZ123456 | Expiry: 31-12-2030 | "
            text += "Issuer: Government Authority | Status: Valid"
            confidence = 0.78  # Image OCR is less confident
            return text, confidence
        except Exception as e:
            print(f"Image extraction error: {e}")
            return "", 0.2
    
    def parse_fields(self, text: str) -> Dict[str, Any]:
        """Parse extracted text into structured fields"""
        fields = {}
        
        # Simple regex-based field extraction
        patterns = {
            'name': r'(?:Name|name)[:=]\s*([A-Za-z\s]+)',
            'pan': r'(?:PAN|pan)[:=]\s*([A-Z0-9]{10})',
            'dob': r'(?:DOB|dob|Date of Birth)[:=]\s*([0-9]{2}-[0-9]{2}-[0-9]{4})',
            'phone': r'(?:Phone|phone)[:=]\s*(\+?[0-9\-]+)',
            'address': r'(?:Address|address)[:=]\s*([A-Za-z0-9\s,]+)',
            'id_number': r'(?:ID|id)[:=]\s*([A-Z0-9]+)',
        }
        
        for field_name, pattern in patterns.items():
            match = re.search(pattern, text)
            if match:
                fields[field_name] = match.group(1).strip()
        
        return fields

# ============================================================
# FILE UPLOAD HANDLER
# ============================================================

class DocumentUploadManager:
    """Manages document uploads and tracking"""
    
    def __init__(self, upload_dir: str = "./uploads"):
        self.upload_dir = Path(upload_dir)
        self.upload_dir.mkdir(parents=True, exist_ok=True)
        self.documents: Dict[str, DocumentFile] = {}
    
    def upload_document(self, file_path: str, doc_type: DocumentType = DocumentType.UNKNOWN) -> DocumentFile:
        """Register and track uploaded document"""
        source_path = Path(file_path)
        
        if not source_path.exists():
            raise FileNotFoundError(f"File not found: {file_path}")
        
        # Create document record
        file_id = str(uuid.uuid4())[:8]
        doc = DocumentFile(
            file_id=file_id,
            file_name=source_path.name,
            file_path=str(source_path),
            doc_type=doc_type,
            file_format=source_path.suffix.lower().strip('.')
        )
        
        self.documents[file_id] = doc
        return doc
    
    def get_documents_by_type(self, doc_type: DocumentType) -> List[DocumentFile]:
        """Get all documents of specific type"""
        return [d for d in self.documents.values() if d.doc_type == doc_type]
    
    def list_all_documents(self) -> List[DocumentFile]:
        """List all uploaded documents"""
        return list(self.documents.values())

print("✓ Document extraction & upload modules defined")


✓ Document extraction & upload modules defined


In [28]:

# ============================================================
# SPECIALIZED AGENTS (Enhanced with Decision Capability)
# ============================================================

class BaseAgent:
    """Base agent with decision capabilities"""
    
    def __init__(self, name: str):
        self.name = name
        self.last_confidence = 0.0
        self.last_decision = None
    
    def execute(self, case: CaseContext) -> AgentDecision:
        """Execute agent logic and return decision"""
        raise NotImplementedError
    
    def can_recheck(self) -> bool:
        """Can this agent perform re-checks if confidence is low"""
        return True

class CaseIntakeAgent(BaseAgent):
    """Validates case structure and documents"""
    
    def __init__(self):
        super().__init__("CaseIntakeAgent")
    
    def execute(self, case: CaseContext) -> AgentDecision:
        if len(case.documents) == 0:
            return AgentDecision(
                agent_name=self.name,
                decision=DecisionType.REVIEW,
                confidence=0.0,
                reasoning="No documents uploaded"
            )
        
        required_doc_types = [DocumentType.KYC_FORM, DocumentType.ID_PROOF]
        available_types = set(d.doc_type for d in case.documents)
        
        has_all_required = all(req in available_types for req in required_doc_types)
        confidence = 1.0 if has_all_required else 0.6
        
        return AgentDecision(
            agent_name=self.name,
            decision=DecisionType.APPROVE if has_all_required else DecisionType.REVIEW,
            confidence=confidence,
            reasoning=f"Validated {len(case.documents)} documents. Required docs present: {has_all_required}",
            supporting_evidence={"document_count": len(case.documents), "doc_types": [d.doc_type.value for d in case.documents]}
        )

class DocumentExtractionAgent(BaseAgent):
    """Extracts structured data from documents"""
    
    def __init__(self, extractor: DocumentExtractor):
        super().__init__("DocumentExtractionAgent")
        self.extractor = extractor
    
    def execute(self, case: CaseContext) -> AgentDecision:
        extracted_fields = {}
        total_confidence = 0.0
        docs_processed = 0
        
        for doc in case.documents:
            text, conf = self.extractor.extract_from_file(doc.file_path)
            if text:
                doc.extracted_text = text
                doc.extraction_confidence = conf
                fields = self.extractor.parse_fields(text)
                extracted_fields.update(fields)
                total_confidence += conf
                docs_processed += 1
        
        avg_confidence = total_confidence / docs_processed if docs_processed > 0 else 0.0
        case.extracted_data["parsed_fields"] = extracted_fields
        
        decision = DecisionType.APPROVE if avg_confidence > 0.7 else DecisionType.REVIEW
        
        return AgentDecision(
            agent_name=self.name,
            decision=decision,
            confidence=avg_confidence,
            reasoning=f"Extracted data from {docs_processed} documents with avg confidence {avg_confidence:.2f}",
            supporting_evidence={
                "extracted_fields": extracted_fields,
                "docs_processed": docs_processed,
                "avg_confidence": avg_confidence
            }
        )

class DocumentVerificationAgent(BaseAgent):
    """Verifies document authenticity and consistency"""
    
    def __init__(self):
        super().__init__("DocumentVerificationAgent")
    
    def execute(self, case: CaseContext) -> AgentDecision:
        extracted = case.extracted_data.get("parsed_fields", {})
        
        # Check for consistency across documents
        issues = []
        confidence = 1.0
        
        if not extracted.get('name'):
            issues.append("Name not extracted")
            confidence -= 0.3
        
        if not extracted.get('pan') and not extracted.get('id_number'):
            issues.append("No valid ID found")
            confidence -= 0.4
        
        decision = DecisionType.APPROVE if confidence > 0.6 else DecisionType.REVIEW
        
        return AgentDecision(
            agent_name=self.name,
            decision=decision,
            confidence=max(0.0, confidence),
            reasoning=f"Document verification: {', '.join(issues) if issues else 'All checks passed'}",
            supporting_evidence={"issues": issues, "documents_checked": len(case.documents)}
        )

class IdentityVerificationAgent(BaseAgent):
    """Cross-checks identity across documents"""
    
    def __init__(self):
        super().__init__("IdentityVerificationAgent")
    
    def execute(self, case: CaseContext) -> AgentDecision:
        extracted = case.extracted_data.get("parsed_fields", {})
        name = extracted.get('name', '')
        
        # Check name consistency across documents
        confidence = 0.95 if name else 0.3
        decision = DecisionType.APPROVE if confidence > 0.8 else DecisionType.REVIEW
        
        return AgentDecision(
            agent_name=self.name,
            decision=decision,
            confidence=confidence,
            reasoning=f"Identity verified as: {name or 'UNKNOWN'}",
            supporting_evidence={"verified_name": name}
        )

class FaceMatchAgent(BaseAgent):
    """Simulates face matching between ID and selfie"""
    
    def __init__(self):
        super().__init__("FaceMatchAgent")
    
    def execute(self, case: CaseContext) -> AgentDecision:
        selfies = [d for d in case.documents if d.doc_type == DocumentType.SELFIE]
        ids = [d for d in case.documents if d.doc_type == DocumentType.ID_PROOF]
        
        if not selfies or not ids:
            return AgentDecision(
                agent_name=self.name,
                decision=DecisionType.REVIEW,
                confidence=0.0,
                reasoning="Missing selfie or ID document"
            )
        
        # Simulated face match
        confidence = 0.94
        decision = DecisionType.APPROVE if confidence > 0.85 else DecisionType.REVIEW
        
        return AgentDecision(
            agent_name=self.name,
            decision=decision,
            confidence=confidence,
            reasoning="Face match successful",
            supporting_evidence={"match_score": confidence}
        )

class ComplianceScreeningAgent(BaseAgent):
    """Screens against PEP, sanctions lists"""
    
    def __init__(self):
        super().__init__("ComplianceScreeningAgent")
        self.pep_list = []
        self.sanctions_list = []
    
    def execute(self, case: CaseContext) -> AgentDecision:
        extracted = case.extracted_data.get("parsed_fields", {})
        name = extracted.get('name', '')
        
        # Simple screening simulation
        is_flagged = False
        confidence = 1.0 if name else 0.5
        
        decision = DecisionType.APPROVE if not is_flagged else DecisionType.REJECT
        
        return AgentDecision(
            agent_name=self.name,
            decision=decision,
            confidence=confidence,
            reasoning="No compliance issues found" if not is_flagged else "Customer flagged",
            supporting_evidence={"pep_match": False, "sanctions_match": False}
        )

print("✓ Specialized agents defined")


✓ Specialized agents defined


In [29]:

# ============================================================
# INTELLIGENT ORCHESTRATOR AGENT (v2.0 - With Decision Logic)
# ============================================================

class IntelligentOrchestratorAgent:
    """Enhanced orchestrator with adaptive decision-making and conflict resolution"""
    
    def __init__(self, llm_config: Dict[str, Any] = None):
        self.llm_config = llm_config or {}
        self.extractor = DocumentExtractor()
        self.upload_manager = DocumentUploadManager()
        
        # Initialize agents
        self.agents = [
            CaseIntakeAgent(),
            DocumentExtractionAgent(self.extractor),
            DocumentVerificationAgent(),
            IdentityVerificationAgent(),
            FaceMatchAgent(),
            ComplianceScreeningAgent(),
        ]
        
        # Decision thresholds (adaptive)
        self.confidence_threshold = 0.7
        self.risk_threshold = 40.0
        self.conflict_threshold = 0.3  # Confidence diff for conflict
        self.max_rechecks = 2
    
    def upload_documents(self, file_paths: List[str], doc_types: List[DocumentType] = None) -> CaseContext:
        """Upload multiple documents and return case context"""
        case = CaseContext(
            case_id=f"KYC-{uuid.uuid4().hex[:12].upper()}",
            created_at=datetime.now(timezone.utc).isoformat()
        )
        
        for i, file_path in enumerate(file_paths):
            doc_type = doc_types[i] if doc_types and i < len(doc_types) else DocumentType.UNKNOWN
            doc = self.upload_manager.upload_document(file_path, doc_type)
            case.documents.append(doc)
            print(f"✓ Uploaded: {doc.file_name} ({doc_type.value})")
        
        return case
    
    def detect_conflicts(self, decisions: List[AgentDecision]) -> List[Dict[str, Any]]:
        """Detect conflicting decisions between agents"""
        conflicts = []
        
        approve_decisions = [d for d in decisions if d.decision == DecisionType.APPROVE]
        reject_decisions = [d for d in decisions if d.decision == DecisionType.REJECT]
        review_decisions = [d for d in decisions if d.decision == DecisionType.REVIEW]
        
        # Detect hard conflicts
        if approve_decisions and reject_decisions:
            conflicts.append({
                "type": "DECISION_CONFLICT",
                "approve_agents": [d.agent_name for d in approve_decisions],
                "reject_agents": [d.agent_name for d in reject_decisions],
                "severity": "HIGH"
            })
        
        # Detect confidence gaps
        if decisions:
            confidences = [d.confidence for d in decisions]
            conf_gap = max(confidences) - min(confidences)
            if conf_gap > self.conflict_threshold:
                conflicts.append({
                    "type": "CONFIDENCE_GAP",
                    "gap": conf_gap,
                    "high_conf_agent": max(decisions, key=lambda d: d.confidence).agent_name,
                    "low_conf_agent": min(decisions, key=lambda d: d.confidence).agent_name,
                    "severity": "MEDIUM"
                })
        
        return conflicts
    
    def request_agent_feedback(self, agent: BaseAgent, case: CaseContext, reason: str) -> AgentDecision:
        """Request agent re-execution with feedback"""
        print(f"  ⚡ Requesting feedback from {agent.name}: {reason}")
        return agent.execute(case)
    
    def resolve_conflicts(self, case: CaseContext, conflicts: List[Dict[str, Any]]) -> None:
        """Attempt to resolve conflicts through agent collaboration"""
        for conflict in conflicts:
            if conflict["type"] == "DECISION_CONFLICT":
                # Request re-checks from uncertain agents
                for decision in case.agent_decisions:
                    if decision.agent_name in conflict["reject_agents"]:
                        agent = next((a for a in self.agents if a.name == decision.agent_name), None)
                        if agent and case.rechecks_performed < self.max_rechecks:
                            new_decision = self.request_agent_feedback(
                                agent, case, "Conflicting decision detected"
                            )
                            case.agent_decisions[-1] = new_decision
                            case.rechecks_performed += 1
    
    def calculate_risk_score(self, decisions: List[AgentDecision]) -> float:
        """Calculate weighted risk score from agent decisions"""
        if not decisions:
            return 50.0
        
        risk_weights = {
            "DocumentExtractionAgent": 0.20,
            "DocumentVerificationAgent": 0.25,
            "IdentityVerificationAgent": 0.15,
            "FaceMatchAgent": 0.15,
            "ComplianceScreeningAgent": 0.25,
        }
        
        total_risk = 0.0
        for decision in decisions:
            # Convert confidence to risk (high confidence = low risk)
            agent_risk = (1.0 - decision.confidence) * 100
            weight = risk_weights.get(decision.agent_name, 0.1)
            total_risk += agent_risk * weight
        
        return min(total_risk, 100.0)
    
    def make_adaptive_decision(self, case: CaseContext) -> Tuple[DecisionType, str]:
        """Make adaptive decision based on all evidence"""
        decisions = case.agent_decisions
        risk_score = case.final_risk_score or 0.0
        
        # Count decisions
        approve_count = sum(1 for d in decisions if d.decision == DecisionType.APPROVE)
        reject_count = sum(1 for d in decisions if d.decision == DecisionType.REJECT)
        review_count = sum(1 for d in decisions if d.decision == DecisionType.REVIEW)
        avg_confidence = sum(d.confidence for d in decisions) / len(decisions) if decisions else 0.0
        
        # Decision logic
        reasoning_parts = []
        
        # Rule 1: Hard reject takes precedence
        if reject_count > 0:
            reasoning_parts.append(f"Reject signals from {reject_count} agent(s)")
            return DecisionType.REJECT, " | ".join(reasoning_parts)
        
        # Rule 2: High confidence across agents
        if avg_confidence >= self.confidence_threshold and approve_count >= len(decisions) - 1:
            reasoning_parts.append(f"High confidence: {avg_confidence:.2f}")
            reasoning_parts.append(f"Approve: {approve_count} agents")
            return DecisionType.APPROVE, " | ".join(reasoning_parts)
        
        # Rule 3: Mixed signals → Review
        if review_count > 0 or avg_confidence < self.confidence_threshold:
            reasoning_parts.append(f"Confidence below threshold: {avg_confidence:.2f}")
            reasoning_parts.append(f"Review signals: {review_count}")
            return DecisionType.REVIEW, " | ".join(reasoning_parts)
        
        # Default
        return DecisionType.REVIEW, "Unable to determine clear decision"
    
    def orchestrate(self, documents: List[str], doc_types: List[DocumentType]) -> CaseContext:
        """Main orchestration flow"""
        # Phase 1: Upload documents
        print("\n" + "="*70)
        print("  AMD INTELLIGENT ORCHESTRATOR - KYC PIPELINE START")
        print("="*70)
        
        case = self.upload_documents(documents, doc_types)
        print(f"\n  Case ID: {case.case_id}")
        print(f"  Documents: {len(case.documents)}")
        
        # Phase 2: Execute agents sequentially
        print("\n" + "-"*70)
        print("  SEQUENTIAL AGENT EXECUTION")
        print("-"*70)
        
        for i, agent in enumerate(self.agents, 1):
            print(f"\n[{i}/{len(self.agents)}] {agent.name}...")
            decision = agent.execute(case)
            case.agent_decisions.append(decision)
            
            # Store evidence
            evidence_item = {
                "agent_name": decision.agent_name,
                "status": "success",
                "confidence": decision.confidence,
                "decision": decision.decision.value,
                "summary": decision.reasoning,
                "risk_contribution": decision.risk_score,
            }
            case.evidence.append(evidence_item)
            
            print(f"  ✓ Decision: {decision.decision.value} | Confidence: {decision.confidence:.2f}")
            print(f"    Reasoning: {decision.reasoning}")
        
        # Phase 3: Conflict detection
        print("\n" + "-"*70)
        print("  CONFLICT DETECTION & RESOLUTION")
        print("-"*70)
        
        conflicts = self.detect_conflicts(case.agent_decisions)
        case.conflicts_detected = conflicts
        
        if conflicts:
            print(f"\n⚠ {len(conflicts)} conflict(s) detected:")
            for conflict in conflicts:
                print(f"  - {conflict['type']}: {conflict['severity']} severity")
            
            self.resolve_conflicts(case, conflicts)
            print(f"\n✓ Resolution attempted (Rechecks: {case.rechecks_performed})")
        else:
            print("\n✓ No conflicts detected")
        
        # Phase 4: Risk calculation & final decision
        print("\n" + "-"*70)
        print("  FINAL RISK ASSESSMENT & DECISION")
        print("-"*70)
        
        risk_score = self.calculate_risk_score(case.agent_decisions)
        case.final_risk_score = risk_score
        print(f"\n  Risk Score: {risk_score:.1f}/100")
        
        final_decision, reasoning = self.make_adaptive_decision(case)
        case.final_decision = final_decision
        case.decision_reasoning = reasoning
        
        print(f"  Final Decision: {final_decision.value}")
        print(f"  Reasoning: {reasoning}")
        
        print("\n" + "="*70)
        print(f"  ✓ PIPELINE COMPLETE - Decision: {final_decision.value}")
        print("="*70)
        
        return case

print("✓ Intelligent Orchestrator Agent defined")


✓ Intelligent Orchestrator Agent defined



# INPUT SECTION - Upload Your Documents Here

## Option 1: Using Real Files (Recommended)
Replace the file paths below with your actual document paths:


In [30]:

# ============================================================
# DOCUMENT UPLOAD CONFIGURATION
# ============================================================

# Example 1: Basic document files
SAMPLE_DOCUMENTS = [
    # Replace these paths with your actual files
    # Supports: .pdf, .jpg, .png, .bmp, .tiff
    "./sample_documents/pan_card.jpg",
    "./sample_documents/adhar_card.jpg",
    "./sample_documents/pic.jpg",
]

SAMPLE_DOC_TYPES = [
    # Specify document types corresponding to above
    # DocumentType.KYC_FORM,
    DocumentType.ID_PROOF,
    DocumentType.ADDRESS_PROOF,
    DocumentType.SELFIE,
]

# ============================================================
# INTERACTIVE DOCUMENT UPLOAD INTERFACE
# ============================================================

def create_sample_documents():
    """Create sample documents for testing (if real files not available)"""
    from pathlib import Path
    
    sample_dir = Path("./sample_documents")
    sample_dir.mkdir(exist_ok=True)
    
    # Create sample text files that simulate documents
    
    kyc_form = sample_dir / "kyc_form.txt"
    kyc_form.write_text(
        "KYC Form\n"
        "Name: John Doe\n"
        "PAN: ABCDE1234F\n"
        "DOB: 01-01-1990\n"
        "Address: 123 Main St, Bangalore\n"
        "Phone: +91-9876543210\n"
    )
    
    id_proof = sample_dir / "id_proof.txt"
    id_proof.write_text(
        "ID Document (Aadhaar)\n"
        "ID Number: 123456789012\n"
        "Name: John Doe\n"
        "DOB: 01-01-1990\n"
        "Status: Valid\n"
    )
    
    return [
        str(kyc_form),
        str(id_proof),
    ]

# Create sample documents
print("Creating sample documents for demonstration...")
sample_paths = create_sample_documents()
print(f"✓ Sample documents created: {sample_paths}")

# Use samples if no real documents provided
documents_to_process = SAMPLE_DOCUMENTS if SAMPLE_DOCUMENTS else sample_paths
doc_types_to_process = SAMPLE_DOC_TYPES if SAMPLE_DOC_TYPES else [
    DocumentType.KYC_FORM,
    DocumentType.ID_PROOF,
]

print(f"\n✓ Ready to process {len(documents_to_process)} document(s)")


Creating sample documents for demonstration...
✓ Sample documents created: ['sample_documents\\kyc_form.txt', 'sample_documents\\id_proof.txt']

✓ Ready to process 3 document(s)



## How to Use the Upload Interface

### Method 1: Using Local File Paths
```python
SAMPLE_DOCUMENTS = [
    "./documents/kyc_form.pdf",
    "./documents/pan_card.jpg",
    "./documents/selfie.png",
]

SAMPLE_DOC_TYPES = [
    DocumentType.KYC_FORM,
    DocumentType.ID_PROOF,
    DocumentType.SELFIE,
]
```

### Method 2: Adding Documents Programmatically
```python
orchestrator = IntelligentOrchestratorAgent()
case = orchestrator.upload_documents(
    file_paths=["path/to/file1.pdf", "path/to/file2.jpg"],
    doc_types=[DocumentType.KYC_FORM, DocumentType.ID_PROOF]
)
```

### Supported Document Types
- `DocumentType.KYC_FORM`: Main KYC application form
- `DocumentType.ID_PROOF`: Government ID (Aadhaar, PAN, Passport, etc.)
- `DocumentType.ADDRESS_PROOF`: Address verification document
- `DocumentType.SELFIE`: Selfie photo for face matching
- `DocumentType.VIDEO_LIVENESS`: Liveness video

### Supported File Formats
- PDF (.pdf)
- Images (.jpg, .jpeg, .png, .bmp, .tiff)



In [31]:

# ============================================================
# EXECUTE INTELLIGENT KYC PIPELINE
# ============================================================

print(f"\nInitializing Intelligent Orchestrator...")
orchestrator = IntelligentOrchestratorAgent(llm_config=LLM_CONFIG)

print(f"\nStarting orchestration with {len(documents_to_process)} document(s)...\n")

# Execute the intelligent KYC pipeline
case_result = orchestrator.orchestrate(
    documents=documents_to_process,
    doc_types=doc_types_to_process
)

print(f"\n\n✓ ORCHESTRATION COMPLETE")
print(f"  Case ID: {case_result.case_id}")
print(f"  Final Decision: {case_result.final_decision.value}")
print(f"  Risk Score: {case_result.final_risk_score:.1f}/100")
print(f"  Confidence Gap: {max([d.confidence for d in case_result.agent_decisions]) - min([d.confidence for d in case_result.agent_decisions]):.3f}")



Initializing Intelligent Orchestrator...

Starting orchestration with 3 document(s)...


  AMD INTELLIGENT ORCHESTRATOR - KYC PIPELINE START
✓ Uploaded: pan_card.jpg (ID_PROOF)
✓ Uploaded: adhar_card.jpg (ADDRESS_PROOF)
✓ Uploaded: pic.jpg (SELFIE)

  Case ID: KYC-24EFBF9A2DD8
  Documents: 3

----------------------------------------------------------------------
  SEQUENTIAL AGENT EXECUTION
----------------------------------------------------------------------

[1/6] CaseIntakeAgent...
  ✓ Decision: REVIEW | Confidence: 0.60
    Reasoning: Validated 3 documents. Required docs present: False

[2/6] DocumentExtractionAgent...
  ✓ Decision: APPROVE | Confidence: 0.78
    Reasoning: Extracted data from 3 documents with avg confidence 0.78

[3/6] DocumentVerificationAgent...
  ✓ Decision: REVIEW | Confidence: 0.30
    Reasoning: Document verification: Name not extracted, No valid ID found

[4/6] IdentityVerificationAgent...
  ✓ Decision: REVIEW | Confidence: 0.30
    Reasoning: Identity ve

In [32]:

# ============================================================
# DETAILED RESULTS ANALYSIS
# ============================================================

print("\n" + "="*70)
print("  DETAILED CASE ANALYSIS")
print("="*70)

# Document Summary
print(f"\n📄 DOCUMENTS PROCESSED ({len(case_result.documents)}):")
for doc in case_result.documents:
    print(f"  - {doc.file_name} ({doc.doc_type.value})")
    print(f"    Extraction Confidence: {doc.extraction_confidence:.2f}")

# Agent Decisions
print(f"\n🤖 AGENT DECISIONS ({len(case_result.agent_decisions)}):")
for decision in case_result.agent_decisions:
    confidence_bar = "█" * int(decision.confidence * 10) + "░" * (10 - int(decision.confidence * 10))
    print(f"\n  {decision.agent_name}")
    print(f"    Decision: {decision.decision.value}")
    print(f"    Confidence: [{confidence_bar}] {decision.confidence:.2f}")
    print(f"    Risk Score: {decision.risk_score:.1f}")
    print(f"    Evidence: {decision.reasoning}")

# Conflicts
print(f"\n⚠️  CONFLICTS DETECTED ({len(case_result.conflicts_detected)}):")
if case_result.conflicts_detected:
    for conflict in case_result.conflicts_detected:
        print(f"  - {conflict['type']} ({conflict['severity']} severity)")
        print(f"    Details: {conflict}")
else:
    print("  None")

# Final Decision
print(f"\n" + "-"*70)
print(f"🎯 FINAL DECISION: {case_result.final_decision.value}")
print(f"   Risk Score: {case_result.final_risk_score:.1f}/100")
print(f"   Reasoning: {case_result.decision_reasoning}")
print("-"*70)



  DETAILED CASE ANALYSIS

📄 DOCUMENTS PROCESSED (3):
  - pan_card.jpg (ID_PROOF)
    Extraction Confidence: 0.78
  - adhar_card.jpg (ADDRESS_PROOF)
    Extraction Confidence: 0.78
  - pic.jpg (SELFIE)
    Extraction Confidence: 0.78

🤖 AGENT DECISIONS (6):

  CaseIntakeAgent
    Decision: REVIEW
    Confidence: [██████░░░░] 0.60
    Risk Score: 0.0
    Evidence: Validated 3 documents. Required docs present: False

  DocumentExtractionAgent
    Decision: APPROVE
    Confidence: [███████░░░] 0.78
    Risk Score: 0.0
    Evidence: Extracted data from 3 documents with avg confidence 0.78

  DocumentVerificationAgent
    Decision: REVIEW
    Confidence: [██░░░░░░░░] 0.30
    Risk Score: 0.0
    Evidence: Document verification: Name not extracted, No valid ID found

  IdentityVerificationAgent
    Decision: REVIEW
    Confidence: [███░░░░░░░] 0.30
    Risk Score: 0.0
    Evidence: Identity verified as: UNKNOWN

  FaceMatchAgent
    Decision: APPROVE
    Confidence: [█████████░] 0.94
    Ris

In [33]:

# ============================================================
# EXPORT CASE DATA & REPORT
# ============================================================

import json

# Convert case to JSON
case_data = {
    "case_id": case_result.case_id,
    "created_at": case_result.created_at,
    "documents": [
        {
            "file_id": d.file_id,
            "file_name": d.file_name,
            "doc_type": d.doc_type.value,
            "extraction_confidence": d.extraction_confidence,
        }
        for d in case_result.documents
    ],
    "agent_decisions": [
        {
            "agent_name": d.agent_name,
            "decision": d.decision.value,
            "confidence": d.confidence,
            "risk_score": d.risk_score,
            "reasoning": d.reasoning,
        }
        for d in case_result.agent_decisions
    ],
    "conflicts_detected": case_result.conflicts_detected,
    "rechecks_performed": case_result.rechecks_performed,
    "final_decision": case_result.final_decision.value,
    "final_risk_score": case_result.final_risk_score,
    "decision_reasoning": case_result.decision_reasoning,
}

# Save to JSON
output_path = Path(f"./case_output_{case_result.case_id}.json")
output_path.write_text(json.dumps(case_data, indent=2))
print(f"✓ Case data exported: {output_path}")

# Display JSON
print(f"\nJSON OUTPUT:")
print(json.dumps(case_data, indent=2)[:1000] + "...")


✓ Case data exported: case_output_KYC-24EFBF9A2DD8.json

JSON OUTPUT:
{
  "case_id": "KYC-24EFBF9A2DD8",
  "created_at": "2026-06-10T12:14:07.449792+00:00",
  "documents": [
    {
      "file_id": "651ac8c3",
      "file_name": "pan_card.jpg",
      "doc_type": "ID_PROOF",
      "extraction_confidence": 0.78
    },
    {
      "file_id": "a55257bf",
      "file_name": "adhar_card.jpg",
      "doc_type": "ADDRESS_PROOF",
      "extraction_confidence": 0.78
    },
    {
      "file_id": "821872a0",
      "file_name": "pic.jpg",
      "doc_type": "SELFIE",
      "extraction_confidence": 0.78
    }
  ],
  "agent_decisions": [
    {
      "agent_name": "CaseIntakeAgent",
      "decision": "REVIEW",
      "confidence": 0.6,
      "risk_score": 0.0,
      "reasoning": "Validated 3 documents. Required docs present: False"
    },
    {
      "agent_name": "DocumentExtractionAgent",
      "decision": "APPROVE",
      "confidence": 0.7799999999999999,
      "risk_score": 0.0,
      "reasoning": "

In [34]:

# ============================================================
# EXPERIMENTAL SCENARIOS (Run different test cases)
# ============================================================

# Scenario 1: Test with different document combinations
# scenario1_docs = [
#     "./documents/kyc_form_high_risk.pdf",
#     "./documents/unverified_id.jpg",
# ]
# result1 = orchestrator.orchestrate(scenario1_docs, [DocumentType.KYC_FORM, DocumentType.ID_PROOF])

# Scenario 2: Test conflict resolution
# scenario2_docs = [
#     "./documents/kyc_form_conflicting.pdf",
#     "./documents/id_different_name.jpg",
# ]
# result2 = orchestrator.orchestrate(scenario2_docs, [DocumentType.KYC_FORM, DocumentType.ID_PROOF])

print("\n✓ Ready for custom scenarios")
print("\nTo test other scenarios, uncomment the code above and modify document paths.")



✓ Ready for custom scenarios

To test other scenarios, uncomment the code above and modify document paths.
